# 06 · 评测与反馈

模块五的短动作窗口 ACT 是初始基线。本模块归档该策略实际闭环失败，按原场景、随机种子重新采集专家恢复示范，加入受保护的训练分组。再训练采用验证过的 32 步动作窗口。最终在未参与反馈的相同新种子上配对比较新旧模型；改善不能单独归因于新增数据。所有轨迹来自 MuJoCo；当前无真机验收。

In [ ]:
import json
import uuid

from examples._display import setup_notebook

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("evaluation_and_feedback")

from physrealx import ArtifactRef, PhysRealX, RunSpec, VersionedRef
from physrealx.domain import TrainingSpec
from physrealx.integrations.act import ACTTrainer
from physrealx.integrations.datasets import dataset_bundle
from physrealx.runtime import load_episode

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
run_key = uuid.uuid4().hex[:12]


## 1. 模型与数据

In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/data_management_and_training/output/results.json").read_text()
)
model_ref = VersionedRef.model_validate(upstream["model"])
evaluation_run = RunSpec.model_validate(upstream["evaluation_run"])
snapshot_ref = ArtifactRef.model_validate(upstream["snapshot"])


## 2. 基线评测

In [ ]:
evaluation_ref = px.evaluations.submit(
    resource_id=f"oven4.act.feedback.{run_key}",
    version="1.0.0",
    model=model_ref,
    run=evaluation_run,
    episode_count=5,
    base_seed=101000,
)
evaluation = px.evaluations.get_report(evaluation_ref)
print("实际基线策略的反馈采集成功率:", evaluation.success_rate)


## 3. 轨迹归档

In [ ]:
from physrealx.integrations.feedback import FailureQuery, RunEvidence

evidence = px.artifacts.put_json(RunEvidence(dataset=evaluation.dataset))
operation = px.telemetry.record(evidence, method="episode_telemetry")
telemetry_ref = operation.output("telemetry")
print("实际 ACT 轨迹已归档，来源与模型可追溯。")


## 4. 失败诊断

In [ ]:
operation = px.failures.analyze(telemetry_ref, method="trajectory_diagnostics")
failure_report_ref = operation.output("failure_report")
failure_report = json.loads(px.artifacts.get_bytes(failure_report_ref))
assert len(failure_report["failures"]) == len(evaluation.failures)
for failure in failure_report["failures"]:
    print(failure["episode_id"], failure["category"], "物体任务目标:", failure["task_goals"])


## 5. 失败数据回流

In [ ]:
selected_ref, selected = None, ()
if failure_report["failures"]:
    query = px.artifacts.put_json(
        FailureQuery(report=failure_report_ref, resource_id=f"oven4.feedback.{run_key}")
    )
    operation = px.feedback.build_dataset(query, method="failure_episodes")
    selected_dataset = dataset_bundle(px.artifacts, operation.output("dataset"))
    selected_ref = px.datasets.register(selected_dataset)
    selected = selected_dataset.episodes
    assert all(not row.success for row in selected)
print("回流真实策略失败数:", len(selected))


## 6. 专家恢复

In [ ]:
recovery_refs, lineage = [], []
for index, summary in enumerate(selected):
    failed = load_episode(px.artifacts, summary)
    recovery_run = RunSpec(
        scene=failed.scene,
        task=failed.task,
        backend=failed.backend,
        policy=evaluation_run.policy,
        episode_count=1,
        base_seed=failed.seed,
        stop_on_success=False,
        dataset_id=f"oven4.recovery.{run_key}.{index}",
        dataset_version="1.0.0",
        record_signals=evaluation_run.record_signals,
        tags=("expert_demonstration", "failure_recovery", failed.episode_id),
    )
    result = px.runs.submit(recovery_run)
    if result.result is None or result.result.dataset is None:
        raise RuntimeError("定向恢复未产生数据集")
    recovered_ref = result.result.dataset
    recovered = px.datasets.get(recovered_ref)
    assert all(row.success for row in recovered.episodes)
    recovery_refs.append(recovered_ref)
    lineage.append(
        {
            "failed_episode": failed.episode_id,
            "recovery_dataset": recovered_ref.model_dump(mode="json"),
            "seed": failed.seed,
        }
    )
print("原始条件下专家接触恢复成功数:", len(recovery_refs))


## 7. 策略再训练

In [ ]:
augmented_ref = px.datasets.training_subset(
    snapshot_ref,
    resource_id=f"oven4.feedback.train.{run_key}",
    augmentations=tuple(recovery_refs),
    required_tag="expert_demonstration",
)
initial_model = px.training.get_model(model_ref)
config = dict(initial_model.training_config["configuration"])
# A longer action horizon carries the policy through the slow approach phase.
# Keep the same optimizer budget; record both data and configuration changes.
config["chunk_size"] = 32
updated_model_ref = px.training.submit(
    TrainingSpec(
        resource_id=f"oven4.act.updated.{run_key}",
        version="1.0.0",
        dataset=augmented_ref,
        trainer=ACTTrainer.descriptor.bind(config=config),
    )
)
updated_model = px.training.get_model(updated_model_ref)
assert updated_model.weights != initial_model.weights
print("实际再训练模型:", updated_model_ref.resource_id, updated_model.metrics)


## 8. 独立对照

In [ ]:
from physrealx.integrations.feedback import episode_metrics

reports, references, contacts = {}, {}, {}
for name, candidate in (("initial", model_ref), ("updated", updated_model_ref)):
    reference = px.evaluations.submit(
        resource_id=f"oven4.act.heldout.{name}.{run_key}",
        version="1.0.0",
        model=candidate,
        run=evaluation_run,
        episode_count=5,
        base_seed=111000,
    )
    report = px.evaluations.get_report(reference)
    reports[name], references[name] = report, reference
    contacts[name] = []
    for summary in px.datasets.get(report.dataset).episodes:
        episode = load_episode(px.artifacts, summary)
        task = px.tasks.get(episode.task)
        metrics = episode_metrics(episode, task)
        contacts[name].append(metrics)
assert reports["updated"].success_rate == 1.0, reports["updated"].model_dump(mode="json")
assert reports["updated"].success_rate >= reports["initial"].success_rate
assert all(
    0 < row["peak_contact_force_n"] < 100 and row["contact_duration_seconds"] > 0
    for row in contacts["updated"]
)
print("相同新种子的独立比较:", {key: value.success_rate for key, value in reports.items()})


## 9. 结果导出

In [ ]:
lineage_ref = px.artifacts.put_json(
    {
        "schema": "physrealx.recovery-lineage/2",
        "links": lineage,
        "initial_model": model_ref.model_dump(mode="json"),
        "updated_model": updated_model_ref.model_dump(mode="json"),
        "training_dataset": augmented_ref.model_dump(mode="json"),
        "paired_success_rates": {key: value.success_rate for key, value in reports.items()},
    },
)

results = {
    "initial_model": model_ref.model_dump(mode="json"),
    "model": updated_model_ref.model_dump(mode="json"),
    "evaluations": {name: ref.model_dump(mode="json") for name, ref in references.items()},
    "failure_report": failure_report_ref.model_dump(mode="json"),
    "feedback_dataset": selected_ref.model_dump(mode="json") if selected_ref else None,
    "training_dataset": augmented_ref.model_dump(mode="json"),
    "recovery_datasets": [ref.model_dump(mode="json") for ref in recovery_refs],
}
results_ref = px.artifacts.export(
    OUTPUT_DIR,
    results,
    files={"failure-report.json": failure_report_ref, "recovery-lineage.json": lineage_ref},
)
print("结果索引:", OUTPUT_DIR / "results.json")
